In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os

import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.lines import Line2D
import geopandas as gpd
from matplotlib.gridspec import GridSpec

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION_"
# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}
vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

In [ ]:
# separate central asia into subregions
rgi_gdf = gpd.read_file(cfg.dataPath / RGI_V6_ROOT /
                        RGI_REGIONS['13']['folder'] /
                        RGI_REGIONS['13']['file'])[['RGIId', 'O2Region']]

glacier_dict_ca = (dfs['13'][['GLACIER', 'RGIId']].drop_duplicates().merge(
    rgi_gdf, on='RGIId', how='left').set_index('GLACIER').to_dict('index'))

dfs['13']['O2Region'] = dfs['13']['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_ca.items()
})

print(dfs['13'].groupby('O2Region').size().rename('n_measurements'))

# Build O2Region -> color mapping
o2_regions = sorted(
    set(v['O2Region'] for v in glacier_dict_ca.values()
        if v['O2Region'] is not None))
region_colors = [
    "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
    "darkgreen", "darkpurple", "pink", "gray", "black"
]
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(o2_regions)
}

# Map each glacier to its O2Region color
color_map = {
    glacier: o2_color_map[info['O2Region']]
    for glacier, info in glacier_dict_ca.items()
    if info['O2Region'] is not None
}

m = plot_stakes_folium(dfs['13'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
deduped = dfs['01'][['GLACIER', 'RGIId']].drop_duplicates()
multi_mask = deduped.groupby('GLACIER')['RGIId'].transform('count') > 1

deduped = deduped.copy()
deduped['GLACIER_NEW'] = deduped['GLACIER']
deduped.loc[multi_mask, 'GLACIER_NEW'] = (
    deduped.loc[multi_mask, 'GLACIER'] + '_' +
    deduped.loc[multi_mask, 'RGIId'].str.extract(r'\.(\d+)$')[0])
# EASTFORK_01 → EASTFORK_01_00022, EASTFORK_01_00021

dfs['01'] = dfs['01'].merge(deduped[['GLACIER', 'RGIId', 'GLACIER_NEW']],
                            on=['GLACIER', 'RGIId'],
                            how='left')
dfs['01']['GLACIER'] = dfs['01']['GLACIER_NEW'].fillna(dfs['01']['GLACIER'])
dfs['01'] = dfs['01'].drop(columns='GLACIER_NEW')

# separate Alaska into subregions
rgi_gdf = gpd.read_file(cfg.dataPath / RGI_V6_ROOT /
                        RGI_REGIONS['01']['folder'] /
                        RGI_REGIONS['01']['file'])[['RGIId', 'O2Region']]

glacier_dict_alaska = (dfs['01'][['GLACIER', 'RGIId']].drop_duplicates().merge(
    rgi_gdf, on='RGIId', how='left').set_index('GLACIER').to_dict('index'))

dfs['01']['O2Region'] = dfs['01']['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_alaska.items()
})

print(dfs['01'].groupby('O2Region').size().rename('n_measurements'))

# Build O2Region -> color mapping
o2_regions = sorted(
    set(v['O2Region'] for v in glacier_dict_alaska.values()
        if v['O2Region'] is not None))
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(o2_regions)
}

# Map each glacier to its O2Region color
color_map = {
    glacier: o2_color_map[info['O2Region']]
    for glacier, info in glacier_dict_alaska.items()
    if info['O2Region'] is not None
}
m = plot_stakes_folium(dfs['01'], color_map=color_map, tooltip_col='O2Region')
m

### Monthly datasets:

In [ ]:
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Step 2: compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

#### Subregions:

In [ ]:
# Separate AT and IT glaciers from the combined dataset for easier handling in experiments
AT_GLACIERS = [
    'GOLDBERG K.',
    'HALLSTAETTER G.',
    'HINTEREIS F.',
    'JAMTAL F.',
    'KESSELWAND F.',
    'KLEINFLEISS K.',
    'OE. WURTEN K.',
    'VENEDIGER K.',
    'VERNAGT F.',
    'ZETTALUNITZ/MULLWITZ K.',
]

IT_GLACIERS = [
    'CAMPO SETT.',
    'CARESER',
    'CARESER CENTRALE',
    'CARESER OCCIDENTALE',
    'CARESER ORIENTALE',
    'CIARDONEY',
    'FONTANA BIANCA / WEISSBRUNNF.',
    'GRAND ETRET',
    'LUNGA (VEDRETTA) / LANGENF.',
    'LUPO',
    'MALAVALLE (VEDR. DI) / UEBELTALF.',
    'PENDENTE (VEDR.) / HANGENDERF.',
    'RIES OCC. (VEDR. DI) / RIESERF. WESTL.',
    'SURETTA MERIDIONALE',
]

data_AT = monthly_cache['IT_AT']['data_monthly'][monthly_cache['IT_AT'][
    'data_monthly']['GLACIER'].isin(AT_GLACIERS)].copy()
data_AT['SOURCE_CODE'] = 'AT'
data_AT_aug = monthly_cache['IT_AT']['data_monthly_aug'][monthly_cache[
    'IT_AT']['data_monthly_aug']['GLACIER'].isin(AT_GLACIERS)].copy()
data_AT_aug['SOURCE_CODE'] = 'AT'

data_IT = monthly_cache['IT_AT']['data_monthly'][monthly_cache['IT_AT'][
    'data_monthly']['GLACIER'].isin(IT_GLACIERS)].copy()
data_IT['SOURCE_CODE'] = 'IT'
data_IT_aug = monthly_cache['IT_AT']['data_monthly_aug'][monthly_cache[
    'IT_AT']['data_monthly_aug']['GLACIER'].isin(IT_GLACIERS)].copy()
data_IT_aug['SOURCE_CODE'] = 'IT'

# drop monthly_cache with 'IT_AT'
monthly_cache['IT'] = {
    'data_monthly': data_IT,
    'data_monthly_aug': data_IT_aug,
    'months_head_pad': monthly_cache['IT_AT']['months_head_pad'],
    'months_tail_pad': monthly_cache['IT_AT']['months_tail_pad']
}
monthly_cache['AT'] = {
    'data_monthly': data_AT,
    'data_monthly_aug': data_AT_aug,
    'months_head_pad': monthly_cache['IT_AT']['months_head_pad'],
    'months_tail_pad': monthly_cache['IT_AT']['months_tail_pad']
}
monthly_cache = {
    key: val
    for key, val in monthly_cache.items() if key != 'IT_AT'
}

print('Measurements per region after splitting IT and AT:')
print(f"Number of measurements in IT: {len(data_IT)}")
print(f"Number of measurements in AT: {len(data_AT)}")


In [ ]:
# --- CENTRAL ASIA subregions ---
data_ca = monthly_cache['CENTRALASIA']['data_monthly'].copy()
data_ca['O2Region'] = data_ca['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_ca.items()
})
data_ca_aug = monthly_cache['CENTRALASIA']['data_monthly_aug'].copy()
data_ca_aug['O2Region'] = data_ca_aug['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_ca.items()
})

data_CA_12 = data_ca[data_ca['O2Region'].isin(['1', '2'])].copy()
data_CA_3 = data_ca[data_ca['O2Region'] == '3'].copy()
data_CA_4 = data_ca[data_ca['O2Region'] == '4'].copy()
data_CA_12_aug = data_ca_aug[data_ca_aug['O2Region'].isin(['1', '2'])].copy()
data_CA_3_aug = data_ca_aug[data_ca_aug['O2Region'] == '3'].copy()
data_CA_4_aug = data_ca_aug[data_ca_aug['O2Region'] == '4'].copy()

for key, dm, dm_aug in [
    ('CA_12', data_CA_12, data_CA_12_aug),
    ('CA_3', data_CA_3, data_CA_3_aug),
    ('CA_4', data_CA_4, data_CA_4_aug),
]:
    dm['SOURCE_CODE'] = key
    dm_aug['SOURCE_CODE'] = key
    monthly_cache[key] = {
        'data_monthly': dm,
        'data_monthly_aug': dm_aug,
        'months_head_pad': monthly_cache['CENTRALASIA']['months_head_pad'],
        'months_tail_pad': monthly_cache['CENTRALASIA']['months_tail_pad'],
    }

print('Measurements per CENTRALASIA subregion:')
for key in ['CA_12', 'CA_3', 'CA_4']:
    print(f"  {key}: {len(monthly_cache[key]['data_monthly'])}")

# --- Alaska subregions ---
data_ala = monthly_cache['ALA']['data_monthly'].copy()
data_ala['O2Region'] = data_ala['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_alaska.items()
})
data_ala_aug = monthly_cache['ALA']['data_monthly_aug'].copy()
data_ala_aug['O2Region'] = data_ala_aug['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict_alaska.items()
})

data_ALA_2 = data_ala[data_ala['O2Region'] == '2'].copy()
data_ALA_4 = data_ala[data_ala['O2Region'] == '4'].copy()
data_ALA_6 = data_ala[data_ala['O2Region'] == '6'].copy()
data_ALA_2_aug = data_ala_aug[data_ala_aug['O2Region'] == '2'].copy()
data_ALA_4_aug = data_ala_aug[data_ala_aug['O2Region'] == '4'].copy()
data_ALA_6_aug = data_ala_aug[data_ala_aug['O2Region'] == '6'].copy()

for key, dm, dm_aug in [
    ('ALA_2', data_ALA_2, data_ALA_2_aug),
    ('ALA_4', data_ALA_4, data_ALA_4_aug),
    ('ALA_6', data_ALA_6, data_ALA_6_aug),
]:
    dm['SOURCE_CODE'] = key
    dm_aug['SOURCE_CODE'] = key
    monthly_cache[key] = {
        'data_monthly': dm,
        'data_monthly_aug': dm_aug,
        'months_head_pad': monthly_cache['ALA']['months_head_pad'],
        'months_tail_pad': monthly_cache['ALA']['months_tail_pad'],
    }

print('\nMeasurements per Alaska subregion:')
for key in ['ALA_2', 'ALA_4', 'ALA_6']:
    print(f"  {key}: {len(monthly_cache[key]['data_monthly'])}")

In [ ]:
# With subregions
TARGET_REGIONS_SUB = [
    'IT', 'SJM', 'CENTRALASIA', 'ALA', 'CA_12', 'CA_3', 'CA_4', 'ALA_2',
    'ALA_4', 'ALA_6'
]

XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Step 3: assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

In [ ]:
# --- inspect padding and FROM_DATE distribution per code ---
for code, entry in monthly_cache.items():
    print(f"\n{code}:")
    print(f"  head_pad: {entry['months_head_pad']}")
    print(f"  tail_pad: {entry['months_tail_pad']}")

    # check FROM_DATE months in the raw dfs
    for rid, df in dfs.items():
        if df is None or "SOURCE_CODE" not in df.columns:
            continue
        df_code = df[df["SOURCE_CODE"].str.upper() == code]
        if len(df_code) == 0:
            continue
        from_dt = pd.to_datetime(df_code["FROM_DATE"].astype(str),
                                 format="%Y%m%d")
        to_dt = pd.to_datetime(df_code["TO_DATE"].astype(str), format="%Y%m%d")
        print(
            f"  FROM_DATE month distribution:\n{from_dt.dt.month.value_counts().sort_index()}"
        )
        print(
            f"  TO_DATE month distribution:\n{to_dt.dt.month.value_counts().sort_index()}"
        )

In [ ]:
# NaN check across all datasets
print("\n--- NaN check ---")
for src_region, res_xreg in res_xreg_by_source.items():
    for split_name, df in [("df_train", res_xreg["df_train"]),
                           ("df_test", res_xreg["df_test"])]:
        n_nan = df.isna().sum().sum()
        print(f"  {src_region} {split_name}: {n_nan} NaNs")

### Domains shifts & feature overlap:

#### Domain shift:

In [ ]:
# EXCLUDE_TARGETS = {"CAW", "ACA", "GRL"}  # set to empty set to include all
EXCLUDE_TARGETS = {}
res_all_xreg_by_source = {}

# The 4 individual targets we always want (each source excludes itself)
ALL_TARGETS = SOURCE_REGIONS + TARGET_REGIONS_SUB

for src_region in SOURCE_REGIONS:
    # Exclude the source region itself from targets
    target_codes = [t for t in ALL_TARGETS if t not in src_region]

    res_all_xreg = build_xreg_res_all(
        res_xreg=res_xreg_by_source[src_region],
        target_source_codes=target_codes,
        source_col="SOURCE_CODE",
        key_prefix=f"XREG_{src_region}_TO",
        ch_code=src_region,
        region_groups={},  # no group regions anymore
    )

    # Filter out excluded targets
    if EXCLUDE_TARGETS:
        res_all_xreg = {
            k: v
            for k, v in res_all_xreg.items()
            if not any(tgt in k for tgt in EXCLUDE_TARGETS)
        }

    res_all_xreg_by_source[src_region] = res_all_xreg
    print(f"{src_region} -> keys:", list(res_all_xreg.keys()))

In [ ]:
RECOMPUTE_SHIFTS = False

CLIMATE_COLS = [
    't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE',
    'POINT_BALANCE'
]
TOPO_COLS = ['aspect', 'slope', 'svf']

SHIFTS_CACHE = BASE_DIR / "domain_shifts_cache.pkl"

if RECOMPUTE_SHIFTS and SHIFTS_CACHE.exists():
    SHIFTS_CACHE.unlink()
    print(f"Deleted existing cache: {SHIFTS_CACHE}")

if SHIFTS_CACHE.exists():
    with open(SHIFTS_CACHE, "rb") as f:
        cache = pickle.load(f)
    scaler_m = cache["scaler_m"]
    scaler_s = cache["scaler_s"]
    blur_m = cache["blur_m"]
    blur_s = cache["blur_s"]
    blur_joint = cache["blur_joint"]
    bandwidths_m = cache["bandwidths_m"]
    bandwidths_s = cache["bandwidths_s"]
    all_shifts_by_source = cache["all_shifts_by_source"]
    print(
        f"Blurs from cache: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )
    print(f"Loaded shifts from cache: {SHIFTS_CACHE}")

else:
    scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
        res_xreg_by_source,
        monthly_cols=CLIMATE_COLS,
        static_cols=TOPO_COLS,
    )

    blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
        res_xreg_by_source,
        monthly_cols=CLIMATE_COLS,
        static_cols=TOPO_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        seed=cfg.seed,
    )
    bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
    bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
    print(
        f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )

    all_shifts_by_source = {}
    for src_region in SOURCE_REGIONS:
        all_shifts = {}
        for key in tqdm(res_all_xreg_by_source[src_region], desc=src_region):
            shift = compute_domain_shift(
                df_src=res_all_xreg_by_source[src_region][key]["df_train"],
                df_tgt=res_all_xreg_by_source[src_region][key]["df_test"],
                monthly_cols=CLIMATE_COLS,
                static_cols=TOPO_COLS,
                scaler_m=scaler_m,
                scaler_s=scaler_s,
                scaler_joint=scaler_joint,
                blur_m=blur_m,
                blur_s=blur_s,
                blur_joint=blur_joint,
                bandwidths_m=bandwidths_m,
                bandwidths_s=bandwidths_s,
                seed=cfg.seed,
            )
            all_shifts[key] = shift
        all_shifts_by_source[src_region] = all_shifts

    with open(SHIFTS_CACHE, "wb") as f:
        pickle.dump(
            {
                "scaler_m": scaler_m,
                "scaler_s": scaler_s,
                "blur_m": blur_m,
                "blur_s": blur_s,
                "blur_joint": blur_joint,
                "bandwidths_m": bandwidths_m,
                "bandwidths_s": bandwidths_s,
                "all_shifts_by_source": all_shifts_by_source,
            }, f)
    print(f"Saved shifts to cache: {SHIFTS_CACHE}")

# Plot summary across regions, one figure per source
for src_region, all_shifts in all_shifts_by_source.items():
    print(f"\nDomain shift summary for source: {src_region}")
    fig = plot_domain_shift_across_regions(all_shifts, src_region=src_region)
    plt.show()

In [ ]:
for key in res_all_xreg:
    Num_measurements = res_all_xreg[key]['df_test']['ID'].nunique()
    print(key, 'Target region num_measurements:', Num_measurements)

In [ ]:
selected_cols = MONTHLY_COLS + [
    c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"
]
selected_cols = selected_cols + ["POINT_BALANCE"]

df_CH = res_all_xreg_by_source['CH']['XREG_CH_TO_NOR']["df_train"]
df_FR = res_all_xreg_by_source['CH']['XREG_CH_TO_FR']["df_test"]
df_ISL = res_all_xreg_by_source['ISL']['XREG_ISL_TO_NOR']["df_train"]
df_NOR = res_all_xreg_by_source['NOR']['XREG_NOR_TO_ISL']["df_train"]

glaciers_to_plot = {
    f"FR": {
        "df": df_FR,
        "color": "#4dac26",  # green
    },
    f"CH ": {
        "df": df_CH,
        "color": "#2166ac",  # blue
    },
    f"ISL": {
        "df": df_ISL,
        "color": "#d6604d",  # red
    },
    f"NOR": {
        "df": df_NOR,
        "color": "#f4a582",  # light red
    },
}

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
    save_prefix="isl_pool_vs_test",
)

## ML models
### Model datasets:

In [ ]:
logs_cache_dir = BASE_DIR / "LSTM_cache"
outputs_xreg_by_source = {}

# NaN check on augmented dataframes before building LSTM datasets
print("--- NaN check on augmented dataframes ---")
for src_region, res_xreg in res_xreg_by_source.items():
    for split_name, df in [("df_train_aug", res_xreg["df_train_aug"]),
                           ("df_test_aug", res_xreg["df_test_aug"])]:
        feat_cols = [c for c in MONTHLY_COLS + STATIC_COLS if c in df.columns]
        n_nan_feat = df[feat_cols].isna().sum().sum()
        n_nan_tgt = df["POINT_BALANCE"].isna().sum(
        ) if "POINT_BALANCE" in df.columns else "N/A"
        print(
            f"  {src_region} {split_name}: feature NaNs={n_nan_feat}, target NaNs={n_nan_tgt}"
        )

for src_region in SOURCE_REGIONS:
    target_codes = [
        t for t in SOURCE_REGIONS + TARGET_REGIONS_SUB if t != src_region
    ]

    outputs_xreg_by_source[src_region] = build_or_load_lstm_all_xreg(
        cfg=cfg,
        res_xreg=res_xreg_by_source[src_region],
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=logs_cache_dir / src_region,
        ch_code=src_region,
        target_source_codes=target_codes,
        region_groups={},  # no group regions anymore
    )
    print(f"{src_region} -> LSTM dataset keys:",
          list(outputs_xreg_by_source[src_region].keys()))

    # NaN/Inf check on the resulting datasets
    print(f"--- NaN/Inf check for {src_region} ---")
    for exp_key, assets in outputs_xreg_by_source[src_region].items():
        for ds_name in ["ds_train", "ds_test"]:
            ds = assets.get(ds_name)
            if ds is None:
                continue
            # Check directly on the underlying tensors (no iteration needed)
            checks = {"x_m": ds.Xm, "x_s": ds.Xs, "y": ds.y}
            any_issue = False
            for name, t in checks.items():
                n_nan = torch.isnan(t).sum().item()
                n_inf = torch.isinf(t).sum().item()
                if n_nan > 0 or n_inf > 0:
                    print(
                        f"  {exp_key} {ds_name} {name}: NaNs={n_nan}, Infs={n_inf}"
                    )
                    any_issue = True
            if not any_issue:
                print(f"  {exp_key} {ds_name}: OK")

### Train model:

#### LSTM:

In [ ]:
default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0,
    'loss_spec': None
}
models_by_source_lstm = {}
infos_by_source_lstm = {}

TRAIN_FLAG = True  # set to True to retrain from scratch
MODEL_DATE = "2026-05-18"  # pin date so filenames are stable across runs

for src_region in SOURCE_REGIONS:
    # Pick any key to get the train assets — they're the same across all targets
    any_key = next(iter(outputs_xreg_by_source[src_region]))
    train_assets = outputs_xreg_by_source[src_region][any_key]

    model, model_path, info = train_or_load_one_source_model(
        cfg=cfg,
        key=src_region,
        lstm_assets=train_assets,
        best_params=default_params,
        device=device,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"lstm_xreg_{src_region}",
        train_flag=TRAIN_FLAG,
        force_retrain=False,
        epochs=150,
        date=MODEL_DATE,
        model_type="lstm",
    )

    models_by_source_lstm[src_region] = model
    infos_by_source_lstm[src_region] = {
        "model_path": model_path,
        **(info or {})
    }
    print(f"{src_region} -> model trained/loaded from {model_path}")

In [ ]:
df_metrics_by_source_lstm = {}
preds_by_source_lstm = {}
figs_by_source_lstm = {}

# Preferred display order for target regions — any not listed appear at the end
DISPLAY_ORDER = [
    'CH',
    'FR',
    'IT'
    'NOR',
    'ISL',
    'SJM',
    'ALA',
    'CENTRALASIA',
]

EXCLUDE_TARGETS = {'CENTRALASIA', 'ALA'}
for src_region in SOURCE_REGIONS:
    print(f"\nEvaluating source region: {src_region}")
    target_keys = [
        k for k in outputs_xreg_by_source[src_region].keys()
        if k.split("_TO_")[-1] not in EXCLUDE_TARGETS
    ]
    models_by_key = {
        key: models_by_source_lstm[src_region]
        for key in target_keys
    }

    # Derive custom_order from actual keys, sorted by preferred display order
    available_targets = [k.split("_TO_")[-1] for k in target_keys]
    custom_order = [t for t in DISPLAY_ORDER if t in available_targets]
    # append any targets not in DISPLAY_ORDER at the end
    custom_order += [t for t in available_targets if t not in custom_order]

    df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models(
        cfg=cfg,
        models_by_key=models_by_key,
        lstm_assets_by_key=outputs_xreg_by_source[src_region],
        device=device,
        save_dir=f"figures/eval_xreg/{src_region}",
        ax_xlim=(-16, 10),
        ax_ylim=(-16, 10),
        grid_shape=(4, 3),
        grid_figsize=(25, 12),
        domain_shifts=all_shifts_by_source[src_region],
        complement_key=f'XREG_{src_region}_TO_',
        custom_order=custom_order,
    )

    df_metrics_by_source_lstm[src_region] = df_metrics
    preds_by_source_lstm[src_region] = preds_by_key
    figs_by_source_lstm[src_region] = figs_by_key

#### Transformer:

In [ ]:
gs_logs_dir = Path(cfg.dataPath) / path_cache / "TF_REGION/logs/Transformer_GS"
logs_gs_transformer = pd.read_csv(
    os.path.join(gs_logs_dir, "transformer_gs_pooled_2026-05-20.csv"))
logs_gs_transformer.sort_values("valid_loss")

In [ ]:
# best by validation loss
best_row = logs_gs_transformer.sort_values("valid_loss").iloc[1]
print("Best config:")
print(best_row[[
    "d_model", "nhead", "num_layers", "dim_feedforward", "dropout",
    "static_layers", "static_hidden", "static_dropout", "head_dropout", "lr",
    "weight_decay", "valid_loss", "val_rmse_a", "val_rmse_w"
]])

best_params_gs = {
    "Fm":
    int(best_row["Fm"]),
    "Fs":
    int(best_row["Fs"]),
    "d_model":
    int(best_row["d_model"]),
    "nhead":
    int(best_row["nhead"]),
    "num_layers":
    int(best_row["num_layers"]),
    "dim_feedforward":
    int(best_row["dim_feedforward"]),
    "dropout":
    float(best_row["dropout"]),
    "static_layers":
    int(best_row["static_layers"]),
    "static_hidden": (None if pd.isna(best_row["static_hidden"]) else int(
        best_row["static_hidden"])),
    "static_dropout": (None if pd.isna(best_row["static_dropout"]) else float(
        best_row["static_dropout"])),
    "head_dropout":
    float(best_row["head_dropout"]),
    "lr":
    float(best_row["lr"]),
    "weight_decay":
    float(best_row["weight_decay"]),
    "two_heads":
    False,
    "loss_spec":
    None,
    "T_max":
    32,
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")

In [ ]:
transformer_params = best_params_gs

models_by_source_transformer = {}
infos_by_source_transformer = {}

TRAIN_FLAG = True
MODEL_DATE = "2026-05-29"

for src_region in SOURCE_REGIONS:
    any_key = next(iter(outputs_xreg_by_source[src_region]))
    train_assets = outputs_xreg_by_source[src_region][any_key]

    model, model_path, info = train_or_load_one_source_model(
        cfg=cfg,
        key=src_region,
        lstm_assets=train_assets,
        best_params=transformer_params,
        device=device,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"transformer_xreg",  # <-- changed
        train_flag=TRAIN_FLAG,
        force_retrain=False,
        epochs=150,
        date=MODEL_DATE,
        model_type="transformer",  # <-- new
    )

    models_by_source_transformer[src_region] = model
    infos_by_source_transformer[src_region] = {
        "model_path": model_path,
        **(info or {})
    }
    print(f"{src_region} -> model trained/loaded from {model_path}")

In [ ]:
df_metrics_by_source_transformer = {}
preds_by_source_transformer = {}
figs_by_source_transformer = {}

# Preferred display order for target regions — any not listed appear at the end
for src_region in SOURCE_REGIONS:
    print(f"\nEvaluating source region: {src_region}")
    target_keys = [
        k for k in outputs_xreg_by_source[src_region].keys()
        if k.split("_TO_")[-1] not in EXCLUDE_TARGETS
    ]
    models_by_key = {
        key: models_by_source_transformer[src_region]
        for key in target_keys
    }

    # Derive custom_order from actual keys, sorted by preferred display order
    available_targets = [k.split("_TO_")[-1] for k in target_keys]
    custom_order = [t for t in DISPLAY_ORDER if t in available_targets]
    # append any targets not in DISPLAY_ORDER at the end
    custom_order += [t for t in available_targets if t not in custom_order]

    df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models(
        cfg=cfg,
        models_by_key=models_by_key,
        lstm_assets_by_key=outputs_xreg_by_source[src_region],
        device=device,
        save_dir=f"figures/eval_xreg/{src_region}",
        ax_xlim=(-16, 10),
        ax_ylim=(-16, 10),
        grid_shape=(4, 3),
        grid_figsize=(25, 12),
        domain_shifts=all_shifts_by_source[src_region],
        complement_key=f'XREG_{src_region}_TO_',
        custom_order=custom_order,
    )

    df_metrics_by_source_transformer[src_region] = df_metrics
    preds_by_source_transformer[src_region] = preds_by_key
    figs_by_source_transformer[src_region] = figs_by_key

### Comparison of models:

In [ ]:
outputs_xreg_by_source['CH'].keys()

In [ ]:
src = "CH"
targets = ["FR", "ISL", "ALA_4", "CA_3"]  # adjust as needed

ncols = len(targets)
fig, axes = plt.subplots(
    2,
    ncols,
    figsize=(14, (90 * 2) / 25.4),
    sharex=True,
    sharey=True,
)

for row_i, (model_label, models_by_source) in enumerate([
    ("LSTM", models_by_source_lstm),
    ("Transformer", models_by_source_transformer),
]):
    model = models_by_source[
        src]  # one model trained on src, applied to all targets

    for col_i, tgt in enumerate(targets):
        ax = axes[row_i, col_i]
        key = f"XREG_{src}_TO_{tgt}"

        out, test_df_preds, created_fig, ax = evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key=outputs_xreg_by_source[src][key],
            ax=ax,
            ax_xlim=(-16, 10),
            ax_ylim=(-16, 10),
            title=f"{src} → {tgt}",
            legend_fontsize=NATURE_SPECS["font_min_pt"],
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        
        panel_label = chr(ord('a') + row_i * ncols + col_i)
        ax.text(
            0.02, 0.98,
            f"({panel_label})",
            transform=ax.transAxes,
            fontsize=NATURE_SPECS["font_max_pt"],
            fontweight="bold",
            va="top",
            ha="left",
        )
        
        leg = ax.get_legend()
        leg.remove()

        if col_i == 0:
            ax.set_ylabel(
                f"{model_label}\nmodeled PMB (m w.e.)",
                fontsize=NATURE_SPECS["font_max_pt"],
            )
        else:
            ax.set_ylabel("")

        if row_i == 0:
            ax.set_xlabel("")

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"],
)
fig.suptitle(
    f"Zero-shot transfer — source: {src}  |  LSTM (top) vs Transformer (bottom)",
    fontsize=NATURE_SPECS["font_max_pt"]+2,
    y=1.01,
)
fig.tight_layout(rect=[0, 0.05, 1, 0.98])

fig.savefig(f"figures/paperTF/zeroshot_{src}_lstm_vs_transformer.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

In [ ]:
LSTM_COLOR = NATURE_PALETTE["sky_blue"]
TF_COLOR = NATURE_PALETTE["vermillion"]

METRIC_LABELS = {
    "RMSE_annual": "RMSE annual\n(m w.e.)",
    "RMSE_winter": "RMSE winter\n(m w.e.)",
    "RMSE_summer": "RMSE summer\n(m w.e.)",
    "MAE_annual": "MAE annual\n(m w.e.)",
    "R2_annual": "R² annual",
    "R2_winter": "R² winter",
    "pearson_annual": "Pearson r annual",
}

SOURCE_ORDER = ["CH", "FR", "NOR", "ISL", "ALA", "SJM"]
TARGET_ORDER = [
    "CH",
    "FR",
    "IT",
    "NOR",
    "ISL",
    "ALA",
    "SJM",
]


def _parse_key(key):
    m = re.match(r"XREG_(.+)_TO_(.+)", key)
    return (m.group(1), m.group(2)) if m else (key, key)


def _sort_key(val, order):
    try:
        return order.index(val)
    except ValueError:
        return len(order)


# ── inputs ────────────────────────────────────────────────────────────────────
metrics = ["RMSE_annual", "RMSE_winter"]  # adjust as needed
exclude_targets = []  # adjust as needed
save_path = None  # e.g. "figures/paperTF/zeroshot_comparison.png"

# ── build long DataFrame from both dicts ──────────────────────────────────────
rows = []
for model_label, metrics_dict in [("LSTM", df_metrics_by_source_lstm),
                                  ("Transformer",
                                   df_metrics_by_source_transformer)]:
    for src, df in metrics_dict.items():
        for key, row in df.iterrows():
            _, tgt = _parse_key(key)
            if tgt in exclude_targets:
                continue
            entry = {"source": src, "target": tgt, "model": model_label}
            entry.update({m: row[m] for m in metrics if m in row.index})
            rows.append(entry)

import pandas as pd

df_all = pd.DataFrame(rows)

sources = sorted(df_all["source"].unique(),
                 key=lambda s: _sort_key(s, SOURCE_ORDER))
targets_per_source = {
    src:
    sorted(df_all[df_all["source"] == src]["target"].unique(),
           key=lambda t: _sort_key(t, TARGET_ORDER))
    for src in sources
}

# ── figure ────────────────────────────────────────────────────────────────────
n_metrics, n_sources = len(metrics), len(sources)
mm2in = 1 / 25.4
fig = plt.figure(figsize=(200 * mm2in, n_metrics * 1.55 + 0.9))
gs = GridSpec(n_metrics,
              n_sources,
              figure=fig,
              hspace=0.4,
              wspace=0.38,
              left=0.11,
              right=0.98,
              top=0.91,
              bottom=0.15)

bar_w = 0.34
for row_i, metric in enumerate(metrics):
    vals = df_all[metric].dropna()
    spread = vals.max() - vals.min() or 0.1
    pad = spread * 0.14
    ymin, ymax = vals.min() - pad, vals.max() + pad

    for col_i, src in enumerate(sources):
        ax = fig.add_subplot(gs[row_i, col_i])
        tgts = targets_per_source[src]
        x = np.arange(len(tgts))

        ax.set_facecolor("#f5f5f5")
        for xi in range(0, len(tgts), 2):
            ax.axvspan(xi - 0.5, xi + 0.5, color="white", alpha=1.0, zorder=0)

        for model, color, offset in [("LSTM", LSTM_COLOR, -bar_w / 2),
                                     ("Transformer", TF_COLOR, bar_w / 2)]:
            sub = df_all[(df_all["source"] == src)
                         & (df_all["model"] == model)]
            bar_vals = [
                float(sub.loc[sub["target"] == t, metric].iloc[0])
                if not sub[sub["target"] == t].empty else np.nan for t in tgts
            ]
            ax.bar(x + offset,
                   bar_vals,
                   width=bar_w * 0.92,
                   color=color,
                   alpha=0.7,
                   zorder=3,
                   linewidth=0)

        if any(k in metric for k in ("R2", "r2", "pearson")):
            ax.axhline(0, color="#888", linewidth=0.4, zorder=1)

        ax.set_xlim(-0.5, len(tgts) - 0.5)
        ax.set_ylim(ymin, ymax)
        ax.set_xticks(x)
        ax.set_xticklabels(tgts, fontsize=6, rotation=90, ha="right")
        ax.tick_params(axis="y", labelsize=6, width=0.5, length=2)
        ax.tick_params(axis="x", length=0)
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_linewidth(0.5)

        if row_i == 0:
            ax.set_title(f"Source: {src}",
                         fontsize=7,
                         fontweight="bold",
                         pad=4)
        if col_i == 0:
            ax.set_ylabel(METRIC_LABELS.get(metric, metric),
                          fontsize=6.5,
                          labelpad=4)

fig.legend(
    handles=[
        mpatches.Patch(color=LSTM_COLOR, alpha=0.88, label="LSTM"),
        mpatches.Patch(color=TF_COLOR, alpha=0.88, label="Transformer")
    ],
    loc="lower center",
    ncol=2,
    fontsize=7,
    frameon=False,
    bbox_to_anchor=(0.54, 0.0),
    handlelength=1.2,
    handletextpad=0.4,
)
fig.suptitle(
    "Zero-shot cross-region — LSTM vs Transformer",
    fontsize=8.5,
    #fontweight="bold",
    y=1.003)

if save_path:
    fig.savefig(save_path, dpi=300, bbox_inches="tight")

plt.show()